In [4]:
from workspace.sources.local_datasets.preprocessing.encoders.distillibert import DistillibertEncoder
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.transformations import Truncation
from workspace.sources.models.transformers.bert.distillibert import Distillibert
from sklearn.model_selection import ParameterGrid

import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [5]:
experiment_name = 'prefinal-distillibert-base-uncased'

In [11]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_params_dict,
                              model_hparams_dict):
    dataset_params_grid = ParameterGrid(dataset_params_dict)

    model_hparams_grid = ParameterGrid(model_hparams_dict)

    for dataset_params in dataset_params_grid:
        for model_hparams in model_hparams_grid:
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Distillibert(train_best_model_metric=Loss,
                                 training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()

In [ ]:
pipelines = [PreprocessingPipeline(name='truncated_minimal_bert_pipeline',
                                   iterable=[Truncation(0.1),
                                             DistillibertEncoder()]),
             PreprocessingPipeline(name='truncated_noise_reduction_bert_pipeline',
                                   iterable=[Truncation(0.1), NoiseReduction(),
                                             DistillibertEncoder()])]
dataset_params_dict = {'preprocessings_pipeline': pipelines}

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16, 32],
                                    'learning_rate': [5e-04, 5e-05, 5e-06],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [0.0, 1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

### ReCOVery Dataset

In [7]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

2025/05/14 16:40:17 INFO mlflow.tracking.fluent: Experiment with name 'prefinal-distillibert-base-uncased' does not exist. Creating a new experiment.
2025-05-14 16:40:17,790 - Experiment - INFO - MLflow experiment initialized with ID: 883249264897273672
2025-05-14 16:40:17,791 - Experiment - INFO - Model signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001)
2025-05-14 16:40:17,792 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:40:17,793 - Experiment - INFO - Run signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:40:18,149 - Experiment - INFO - Run I

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Map:   0%|          | 0/142 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2025-05-14 16:40:20,129 - Experiment - INFO - Preprocessing split: val_set
2025-05-14 16:40:20,130 - Experiment - INFO - Preprocessed data does not exist, computing from scratch.


...

2025-05-14 16:44:16,061 - Experiment - INFO - Test metrics: {'test_loss': 0.5189769864082336, 'test_accuracy': 0.7333333333333333, 'test_precision': 0.875, 'test_recall': 0.7, 'test_f1_score': 0.7777777777777778, 'test_roc_auc': 0.86, 'test_false_positives_rate': 0.2, 'test_false_negatives_rate': 0.3, 'test_runtime': 0.5339, 'test_samples_per_second': 56.188, 'test_steps_per_second': 7.492, 'test_epoch': 5.0}
2025-05-14 16:44:16,083 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:44:16,085 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:44:16,165 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:44:16,167 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-14 16:44:16,168 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-36
2025-05-14 16:44:16,433 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.4238967299461365, 'eval_accuracy': 0.8666666666666

2025-05-14 16:44:16,955 - Experiment - INFO - Test metrics: {'test_loss': 0.6350271105766296, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.755, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5163, 'test_samples_per_second': 58.11, 'test_steps_per_second': 7.748, 'test_epoch': 2.0}
2025-05-14 16:44:16,977 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:44:16,979 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:44:17,122 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:44:17,123 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-14 16:44:17,124 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-18
2025-05-14 16:44:17,380 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.8038100004196167, 'eval_accuracy': 0.133

C:\Users\nikita.bardatskii\git\bi-bap-2025-bardanik\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2025-05-14 16:44:17,898 - Experiment - INFO - Test metrics: {'test_loss': 0.7496827840805054, 'test_accuracy': 0.3333333333333333, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1_score': 0.0, 'test_roc_auc': 0.545, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 1.0, 'test_runtime': 0.5112, 'test_samples_per_second': 58.684, 'test_steps_per_second': 7.825, 'test_epoch': 1.0}
2025-05-14 16:44:17,921 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:44:17,923 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:44:18,010 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:

2025-05-14 16:44:18,760 - Experiment - INFO - Test metrics: {'test_loss': 0.6350271105766296, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.755, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5097, 'test_samples_per_second': 58.856, 'test_steps_per_second': 7.847, 'test_epoch': 2.0}
2025-05-14 16:44:18,784 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:44:18,787 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:44:18,865 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:44:18,866 - Experiment - INFO - Evaluating model using loss metric...
2025-05-14 16:44:18,866 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-36
2025-05-14 16:44:19,112 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.4238967299461365, 'eval_accuracy': 0.8666666666666667, 

2025-05-14 16:44:19,626 - Experiment - INFO - Test metrics: {'test_loss': 0.6350271105766296, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.755, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5087, 'test_samples_per_second': 58.975, 'test_steps_per_second': 7.863, 'test_epoch': 2.0}
2025-05-14 16:44:19,652 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:44:19,654 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:44:19,733 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:44:19,735 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [8]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

2025-05-14 16:44:20,422 - Experiment - INFO - MLflow experiment initialized with ID: 883249264897273672
2025-05-14 16:44:20,423 - Experiment - INFO - Model signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001)
2025-05-14 16:44:20,424 - Experiment - INFO - Dataset signature: dataset(dn=euvsdisinfo,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:44:20,424 - Experiment - INFO - Run signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001);dataset(dn=euvsdisinfo,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:44:21,457 - Experiment - INFO - Run ID: 27761866e9dd4fb98f5cc3c7faa73bb3
2025-05-14 16:44:21,459 - Experiment - INFO - Run name: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2025-05-14 16:44:22,966 - Experiment - INFO - Preprocessing split: val_set
2025-05-14 16:44:22,967 - Experiment - INFO - Preprocessed data does not exist, computing from scratch.


...

2025-05-14 16:50:03,134 - Experiment - INFO - Test metrics: {'test_loss': 0.011721422895789146, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1_score': 1.0, 'test_roc_auc': 1.0, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.55, 'test_samples_per_second': 58.181, 'test_steps_per_second': 7.273, 'test_epoch': 4.0}
2025-05-14 16:50:03,152 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:50:03,152 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:50:03,245 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:50:03,245 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-14 16:50:03,245 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-76
2025-05-14 16:50:03,462 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.0016182662220671773, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.

2025-05-14 16:50:03,998 - Experiment - INFO - Test metrics: {'test_loss': 0.011721422895789146, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1_score': 1.0, 'test_roc_auc': 1.0, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5347, 'test_samples_per_second': 59.845, 'test_steps_per_second': 7.481, 'test_epoch': 4.0}
2025-05-14 16:50:04,016 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:50:04,016 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:50:04,098 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:50:04,098 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-14 16:50:04,098 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-76
2025-05-14 16:50:04,289 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.0016182662220671773, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'ev

2025-05-14 16:50:04,834 - Experiment - INFO - Test metrics: {'test_loss': 0.011721422895789146, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1_score': 1.0, 'test_roc_auc': 1.0, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5293, 'test_samples_per_second': 60.462, 'test_steps_per_second': 7.558, 'test_epoch': 4.0}
2025-05-14 16:50:04,853 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:50:04,853 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:50:04,931 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:50:04,931 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-14 16:50:04,931 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-76
2025-05-14 16:50:05,131 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.0016182662220671773, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1

2025-05-14 16:50:05,679 - Experiment - INFO - Test metrics: {'test_loss': 0.011721422895789146, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1_score': 1.0, 'test_roc_auc': 1.0, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.546, 'test_samples_per_second': 58.613, 'test_steps_per_second': 7.327, 'test_epoch': 4.0}
2025-05-14 16:50:05,695 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:50:05,695 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:50:05,773 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:50:05,781 - Experiment - INFO - Evaluating model using loss metric...
2025-05-14 16:50:05,781 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-76
2025-05-14 16:50:05,995 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.0016182662220671773, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 

2025-05-14 16:50:06,533 - Experiment - INFO - Test metrics: {'test_loss': 0.011721422895789146, 'test_accuracy': 1.0, 'test_precision': 1.0, 'test_recall': 1.0, 'test_f1_score': 1.0, 'test_roc_auc': 1.0, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.538, 'test_samples_per_second': 59.483, 'test_steps_per_second': 7.435, 'test_epoch': 4.0}
2025-05-14 16:50:06,558 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 16:50:06,558 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 16:50:06,633 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 16:50:06,642 - Experiment - INFO - Finished model evaluations stage.


### ISOT Dataset

In [9]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

2025-05-14 16:50:07,876 - Experiment - INFO - MLflow experiment initialized with ID: 883249264897273672
2025-05-14 16:50:07,877 - Experiment - INFO - Model signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001)
2025-05-14 16:50:07,878 - Experiment - INFO - Dataset signature: dataset(dn=isot,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:50:07,878 - Experiment - INFO - Run signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001);dataset(dn=isot,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 16:50:09,036 - Experiment - INFO - Run ID: 02a5ed65982f42f7bc583c5eb5570854
2025-05-14 16:50:09,036 - Experiment - INFO - Run name: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.000

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2025-05-14 16:50:14,088 - Experiment - INFO - Preprocessing split: val_set
2025-05-14 16:50:14,089 - Experiment - INFO - Preprocessed data does not exist, computing from scratch.


...

2025-05-14 17:11:13,849 - Experiment - INFO - Test metrics: {'test_loss': 0.04951540380716324, 'test_accuracy': 0.9933333333333333, 'test_precision': 1.0, 'test_recall': 0.9859154929577465, 'test_f1_score': 0.9929078014184397, 'test_roc_auc': 0.9994651453021929, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.014084507042253521, 'test_runtime': 2.6487, 'test_samples_per_second': 56.632, 'test_steps_per_second': 7.173, 'test_epoch': 7.0}
2025-05-14 17:11:13,871 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:11:13,875 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:11:14,284 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:11:14,284 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-14 17:11:14,285 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-616
2025-05-14 17:11:14,505 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.0035

2025-05-14 17:11:17,166 - Experiment - INFO - Test metrics: {'test_loss': 0.04951540380716324, 'test_accuracy': 0.9933333333333333, 'test_precision': 1.0, 'test_recall': 0.9859154929577465, 'test_f1_score': 0.9929078014184397, 'test_roc_auc': 0.9994651453021929, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.014084507042253521, 'test_runtime': 2.6551, 'test_samples_per_second': 56.496, 'test_steps_per_second': 7.156, 'test_epoch': 7.0}
2025-05-14 17:11:17,195 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:11:17,202 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:11:17,594 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:11:17,595 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-14 17:11:17,596 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-616
2025-05-14 17:11:17,843 - Experiment - INFO - Best entry according to validation metrics: {'eval_l

2025-05-14 17:11:20,461 - Experiment - INFO - Test metrics: {'test_loss': 0.04951540380716324, 'test_accuracy': 0.9933333333333333, 'test_precision': 1.0, 'test_recall': 0.9859154929577465, 'test_f1_score': 0.9929078014184397, 'test_roc_auc': 0.9994651453021929, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.014084507042253521, 'test_runtime': 2.6125, 'test_samples_per_second': 57.416, 'test_steps_per_second': 7.273, 'test_epoch': 7.0}
2025-05-14 17:11:20,488 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:11:20,494 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:11:20,903 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:11:20,904 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-14 17:11:20,904 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-616
2025-05-14 17:11:21,126 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.00354

2025-05-14 17:11:23,799 - Experiment - INFO - Test metrics: {'test_loss': 0.04951540380716324, 'test_accuracy': 0.9933333333333333, 'test_precision': 1.0, 'test_recall': 0.9859154929577465, 'test_f1_score': 0.9929078014184397, 'test_roc_auc': 0.9994651453021929, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.014084507042253521, 'test_runtime': 2.6666, 'test_samples_per_second': 56.251, 'test_steps_per_second': 7.125, 'test_epoch': 7.0}
2025-05-14 17:11:23,826 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:11:23,832 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:11:24,716 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:11:24,718 - Experiment - INFO - Evaluating model using loss metric...
2025-05-14 17:11:24,720 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-616
2025-05-14 17:11:25,063 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.00354946

2025-05-14 17:11:27,690 - Experiment - INFO - Test metrics: {'test_loss': 0.04951540380716324, 'test_accuracy': 0.9933333333333333, 'test_precision': 1.0, 'test_recall': 0.9859154929577465, 'test_f1_score': 0.9929078014184397, 'test_roc_auc': 0.9994651453021929, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.014084507042253521, 'test_runtime': 2.6198, 'test_samples_per_second': 57.256, 'test_steps_per_second': 7.252, 'test_epoch': 7.0}
2025-05-14 17:11:27,717 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:11:27,723 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:11:28,246 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:11:28,249 - Experiment - INFO - Finished model evaluations stage.


### Czech Dataset

In [10]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

2025-05-14 17:11:32,694 - Experiment - INFO - MLflow experiment initialized with ID: 883249264897273672
2025-05-14 17:11:32,696 - Experiment - INFO - Model signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001)
2025-05-14 17:11:32,697 - Experiment - INFO - Dataset signature: dataset(dn=czechdataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 17:11:32,698 - Experiment - INFO - Run signature: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.001);dataset(dn=czechdataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),distillibert_encoder(t=distilbert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 17:11:33,562 - Experiment - INFO - Run ID: 5027a68187e54dcb9de3db6166396351
2025-05-14 17:11:33,563 - Experiment - INFO - Run name: model(mn=distilbert-base-uncased,mmn=loss,e=10,bs=

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2025-05-14 17:11:34,778 - Experiment - INFO - Preprocessing split: val_set
2025-05-14 17:11:34,779 - Experiment - INFO - Preprocessed data does not exist, computing from scratch.


...

2025-05-14 17:15:41,728 - Experiment - INFO - Test metrics: {'test_loss': 0.46790120005607605, 'test_accuracy': 0.8666666666666667, 'test_precision': 1.0, 'test_recall': 0.75, 'test_f1_score': 0.8571428571428571, 'test_roc_auc': 0.9464285714285714, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.25, 'test_runtime': 0.2541, 'test_samples_per_second': 59.025, 'test_steps_per_second': 7.87, 'test_epoch': 5.0}
2025-05-14 17:15:41,744 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:15:41,760 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:15:41,795 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:15:41,795 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-14 17:15:41,795 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-72
2025-05-14 17:15:42,034 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3033319413661957, 'eval_accuracy': 0

2025-05-14 17:15:42,302 - Experiment - INFO - Test metrics: {'test_loss': 0.6047855019569397, 'test_accuracy': 0.8, 'test_precision': 0.7777777777777778, 'test_recall': 0.875, 'test_f1_score': 0.8235294117647058, 'test_roc_auc': 0.9464285714285714, 'test_false_positives_rate': 0.2857142857142857, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.2672, 'test_samples_per_second': 56.145, 'test_steps_per_second': 7.486, 'test_epoch': 8.0}
2025-05-14 17:15:42,324 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:15:42,326 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:15:42,372 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:15:42,372 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-14 17:15:42,375 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-45
2025-05-14 17:15:42,590 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.5762599

2025-05-14 17:15:42,850 - Experiment - INFO - Test metrics: {'test_loss': 0.46790120005607605, 'test_accuracy': 0.8666666666666667, 'test_precision': 1.0, 'test_recall': 0.75, 'test_f1_score': 0.8571428571428571, 'test_roc_auc': 0.9464285714285714, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.25, 'test_runtime': 0.2603, 'test_samples_per_second': 57.617, 'test_steps_per_second': 7.682, 'test_epoch': 5.0}
2025-05-14 17:15:42,866 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:15:42,866 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:15:42,913 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:15:42,913 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-14 17:15:42,913 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-72
2025-05-14 17:15:43,135 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3033319413661957, 'eval_accuracy': 0

2025-05-14 17:15:43,391 - Experiment - INFO - Test metrics: {'test_loss': 0.6047855019569397, 'test_accuracy': 0.8, 'test_precision': 0.7777777777777778, 'test_recall': 0.875, 'test_f1_score': 0.8235294117647058, 'test_roc_auc': 0.9464285714285714, 'test_false_positives_rate': 0.2857142857142857, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.2554, 'test_samples_per_second': 58.73, 'test_steps_per_second': 7.831, 'test_epoch': 8.0}
2025-05-14 17:15:43,411 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:15:43,417 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:15:43,460 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:15:43,462 - Experiment - INFO - Evaluating model using loss metric...
2025-05-14 17:15:43,462 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-72
2025-05-14 17:15:43,709 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.3033319413661957, 'eval_

2025-05-14 17:15:43,967 - Experiment - INFO - Test metrics: {'test_loss': 0.6047855019569397, 'test_accuracy': 0.8, 'test_precision': 0.7777777777777778, 'test_recall': 0.875, 'test_f1_score': 0.8235294117647058, 'test_roc_auc': 0.9464285714285714, 'test_false_positives_rate': 0.2857142857142857, 'test_false_negatives_rate': 0.125, 'test_runtime': 0.2543, 'test_samples_per_second': 58.997, 'test_steps_per_second': 7.866, 'test_epoch': 8.0}
2025-05-14 17:15:43,983 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 17:15:43,983 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 17:15:44,031 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 17:15:44,031 - Experiment - INFO - Finished model evaluations stage.
